<a href="https://colab.research.google.com/github/mariacgomez-tech/Analitica-de-Negocios/blob/main/2_Arbol_de_Decision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Caso de Estudio**
Una entidad financiera (FINTECH) quiere implementar un modelo de árbol de decision para mejorar la preaprobación de créditos de consumo de sus solicitantes de este tipo de créditos. Para este proceso vamos a utilizar las variables:

*Edad: Indica el número de años que posee una persona, o el tiempo que usted lleva en el sistema financiero.
*Ingresos: Engloba todos los ingresos que recibe una persona además si posee salio mensual (USD)
* Egresos:
* Monto:

0. Se procesde con la carga de las librerías de trabajo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

1. Se procede con la carga de los datos de trabajo.

In [ ]:
nxl= '/content/sample_data/1. SolicitantesCrédito(USD) (3).xlsx'
XDB= pd.read_excel(nxl, sheet_name= 1)

XD=XDB.iloc[:,[1,10,11,25]]
yd=XDB.iloc[:,32]
display(XD)


,Edad,Ingresos,Egresos,Monto (EAD)
0,38,1356.14400,1685.622667,625.562230
1,51,286.01600,364.462000,140.031984
2,37,517.46325,629.208889,284.564492
3,29,473.27000,551.616889,309.647588
4,42,750.09175,806.715778,500.663578
...,...,...,...,...
5837,48,1207.84800,753.801111,748.041791
5838,31,1472.77200,953.812889,870.793819
5839,38,773.01975,672.910667,594.947894
5840,43,635.50175,780.691556,305.580539


2. Se procede con la implementación del modelo de árbol

In [ ]:
mar=DecisionTreeClassifier(criterion='gini', max_depth=4)
mar.fit(XD,yd) #Aquí se ajusta el modelo. Busca relación entrada-salida

# Y qué fue lo que hizo el modelo?
ydp=mar.predict(XD) #Esto es lo que pronostica el modelo

#Se construye la matriz de confusión
cm=confusion_matrix(yd,ydp)
display(cm)
VN=cm[0,0]; FP=cm[0,1]; FN=cm[1,0]; VP=cm[1,1]

#Métricas de desempeño
Ex=(VP+VN)/len(XD) #1. Exactitud: Comportamiento general
print('La Exactitud es: ',Ex)
Sen=VP/(VP+FN) #2. Sensibilidad: Como se comporta pronosticando PreAprobados
print('La Sensibilidad es: ',Sen)
Spe=VN/(VN+FP) #3. Especificidad: Como se comporta frente a los negativos
print('La Especificidad es: ',Spe)
Pre=VP/(VP+FP) #4. Precisión: Como se comporta pronosticando PreAprobados
print('La Precisión es: ',Pre)
PreNeg=VN/(VN+FN) #5. Precisión Negativa: Como se comporta pronosticando Negativos
print('La Precisión Negativa es: ',PreNeg)

array([[2301,  658],
       [ 644, 2239]])

La Exactitud es:  0.7771311194796303
La Sensibilidad es:  0.7766215747485259
La Especificidad es:  0.7776275768840825
La Precisión es:  0.772868484639282
La Precisión Negativa es:  0.7813242784380305


3. Despliegue del árbol de decisión

In [ ]:
from sklearn.tree import export_graphviz #Exporta los datos a un gráfico
from pydotplus import graph_from_dot_data #Es un graficador

vs=["Edad", "Ingresos", "Egresos", "Monto"] #Títulos del árbol
dot_data=export_graphviz(mar, feature_names=vs) #Exportar de números a gráfico en pdf
graph=graph_from_dot_data(dot_data) #Graficamo el árbol
graph.write_png('arbol.png') #Guardamos el árbol en pdf

True

**Análisis de resultados**
De la base de datos se puede observar un total de 2959 solicitantes de crédito que poseen la categoría de prenegados, mientras que para la categoría de preaprobados la matriz cuenta con uun total de 2883 créditos preaprobados. De los 2959 datos, el modelo pronosticó correctamente un total de 2301 datos. De los 2883 preaprobados el modelo pronosticó correctamente un total de 2239. El modelo logra identificar en un 77% los créditos prenegados (2301/2959), mientras que el modelo logró identificar en un 77.66% los créditos preaprobados (2239/2883).

Con respecto a las métrica, podemos observar que el modelo logró una exactitud de 77.71%, lo que indica el buen comportamiento general del modelo frente a la clasificación de créditos en las dos categorías de preaprobación. Se destacan la especificidad y la precisión, las cuales lograron valores en promedio por encima del 77%, lo que supera el límite inferior definido para este tipo de modelos de clasificaciín el cual se ubica en el 75%.

De acuerdo con el árbol de decisión se destaca un nodo puro (10/0) el cual posee la siguiente regla de decisión: si una persona cumple con esta SI Monto<=322 and Ing>376 and Egr<=685 and Ing>644 (923/5), tendrá una probabilidad de aprobación del 100%. Es importante destacar que los nodos puros poseen un gini=0 (El modelo diferencia muy bien los buenapagas)

A pesar de la no existencia de más nodos puros, se destacan los nodos extremos o las reglas del negocio extremas. Se destaca una segunda regla que logra un porcentaje de aprobación del 99%, y la cual posee la siguiente estructura: Si Monto<=322 and Ing<=376 and Monto<=178 and Ing<=232. Por su parte, el nodo derecho posee un porcentaje de negación del 98%, en donde la regla se define: Monto>322 and Ing>896 and Monto>961 and Ing>1178 (7/576)